In [56]:
import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

In [57]:

from pathlib import Path

# Diretório atual de execução
CURRENT_DIR = Path.cwd().resolve()

# Identifica a raiz do projeto
if CURRENT_DIR.name == "agg" and CURRENT_DIR.parent.name == "out":
    PROJECT_DIR = CURRENT_DIR.parent.parent
elif CURRENT_DIR.name == "notebooks":
    PROJECT_DIR = CURRENT_DIR.parent
else:
    PROJECT_DIR = CURRENT_DIR

# Diretório dos arquivos de entrada
BASE_DIR = PROJECT_DIR / "out" / "agg"

PR_FILE = BASE_DIR / "pr_slim.csv"
REPOS_FILE = BASE_DIR / "repos.csv"
META_FILE = BASE_DIR / "meta.json"

# Diretório correto de saída
OUTPUT_DIR = PROJECT_DIR / "public" / "data"

NETWORK_TOP = 100

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Diretório atual:", CURRENT_DIR)
print("Diretório raiz:", PROJECT_DIR)
print("Diretório dos dados:", BASE_DIR)
print("Diretório de saída:", OUTPUT_DIR)

for file in [PR_FILE, REPOS_FILE, META_FILE]:
    print(file.name, ":", file.exists())
    if not file.exists():
        raise FileNotFoundError(file)


Diretório atual: C:\Users\fpaul\Documents\GitHub\dataviz-final-work\out\agg
Diretório raiz: C:\Users\fpaul\Documents\GitHub\dataviz-final-work
Diretório dos dados: C:\Users\fpaul\Documents\GitHub\dataviz-final-work\out\agg
Diretório de saída: C:\Users\fpaul\Documents\GitHub\dataviz-final-work\public\data
pr_slim.csv : True
repos.csv : True
meta.json : True


In [58]:
def json_safe(value: Any) -> Any:
    """
    Converte valores do Pandas e NumPy para tipos compatíveis
    com JSON, tratando valores ausentes.
    """

    if value is None or value is pd.NA:
        return None

    if isinstance(value, dict):
        return {
            str(k): json_safe(v)
            for k, v in value.items()
        }

    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]

    if isinstance(value, np.integer):
        return int(value)

    if isinstance(value, np.floating):
        return None if not np.isfinite(value) else float(value)

    if isinstance(value, np.bool_):
        return bool(value)

    if isinstance(value, pd.Timestamp):
        return value.strftime("%Y-%m-%d")

    if isinstance(value, float) and not np.isfinite(value):
        return None

    if pd.isna(value):
        return None

    return value


def write_json(path: Path, data: Any) -> None:
    """
    Salva os dados em formato JSON.
    """

    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as file:
        json.dump(
            json_safe(data),
            file,
            ensure_ascii=False,
            indent=2,
            allow_nan=False
        )


def records(df: pd.DataFrame) -> list:
    """
    Converte DataFrame para lista de dicionários.
    """

    return json_safe(
        df.astype(object)
        .where(pd.notna(df), None)
        .to_dict(orient="records")
    )


def require_columns(df, required, filename):
    """
    Verifica se todas as colunas necessárias existem.
    """

    missing = sorted(set(required) - set(df.columns))

    if missing:
        raise ValueError(
            f"{filename}: colunas ausentes: {missing}"
        )

In [59]:
# Colunas esperadas nos arquivos
PR_REQUIRED = {
    "id",
    "agent",
    "outcome",
    "ttm_hours",
    "date",
    "week",
    "dow",
    "hour",
    "repo_id",
    "language",
    "stars_bin",
    "n_commits",
    "n_reviews",
    "review_type"
}

REPO_REQUIRED = {
    "repo_id",
    "repo",
    "language",
    "stars",
    "forks",
    "n_pr",
    "n_merged",
    "n_closed",
    "median_ttm_h",
    "pct_human_review",
    "main_agent"
}

# Leitura dos CSVs
pr = pd.read_csv(PR_FILE, low_memory=False)
repos = pd.read_csv(REPOS_FILE, low_memory=False)

# Leitura dos metadados originais
with META_FILE.open("r", encoding="utf-8") as file:
    source_meta = json.load(file)

# Validação
require_columns(pr, PR_REQUIRED, "pr_slim.csv")
require_columns(repos, REPO_REQUIRED, "repos.csv")

# Conversão de tipos
pr["date"] = pd.to_datetime(
    pr["date"],
    errors="coerce"
)

pr["ttm_hours"] = pd.to_numeric(
    pr["ttm_hours"],
    errors="coerce"
)

pr["repo_id"] = pd.to_numeric(
    pr["repo_id"],
    errors="coerce"
)

repos["repo_id"] = pd.to_numeric(
    repos["repo_id"],
    errors="coerce"
)

repos["n_pr"] = pd.to_numeric(
    repos["n_pr"],
    errors="coerce"
).fillna(0)

print("Dados carregados com sucesso!")
print(f"Pull Requests: {len(pr):,}")
print(f"Repositórios: {len(repos):,}")
print(f"Agentes: {pr['agent'].nunique()}")
print(f"Linguagens: {pr['language'].nunique()}")

Dados carregados com sucesso!
Pull Requests: 71,677
Repositórios: 6,673
Agentes: 6
Linguagens: 14


In [60]:
def prepare_monthly_agents(pr):

    data = pr.dropna(
        subset=["date", "agent"]
    ).copy()

    data["month"] = (
        data["date"]
        .dt.to_period("M")
        .astype(str)
    )

    result = (
        data.groupby(
            ["month", "agent"],
            observed=True
        )
        .size()
        .reset_index(name="prs")
        .sort_values(["month", "agent"])
    )

    return records(result)


monthly_agents = prepare_monthly_agents(pr)

write_json(
    OUTPUT_DIR / "monthly_agents.json",
    monthly_agents
)

print("monthly_agents.json gerado!")

monthly_agents.json gerado!


In [61]:
def prepare_outcomes(pr):

    data = pr.dropna(
        subset=["agent", "outcome"]
    ).copy()

    result = (
        data.groupby(
            ["agent", "outcome"],
            observed=True
        )
        .size()
        .reset_index(name="count")
    )

    result["total_agent"] = (
        result.groupby("agent")["count"]
        .transform("sum")
    )

    result["percentage"] = (
        result["count"] /
        result["total_agent"] * 100
    )

    return records(
        result.sort_values(["agent", "outcome"])
    )


outcomes = prepare_outcomes(pr)

write_json(
    OUTPUT_DIR / "outcomes.json",
    outcomes
)

print("outcomes.json gerado!")

outcomes.json gerado!


In [62]:
def prepare_merge_times(pr):

    data = pr.loc[
        (pr["outcome"].astype(str).str.lower() == "merged")
        & pr["ttm_hours"].notna()
        & (pr["ttm_hours"] > 0),
        ["agent", "ttm_hours"]
    ].copy()

    summary = []
    ecdf = []

    quantiles = [
        0.00,
        0.25,
        0.50,
        0.75,
        0.90,
        0.95,
        1.00
    ]

    quantile_names = [
        "min",
        "q1",
        "median",
        "q3",
        "p90",
        "p95",
        "max"
    ]

    for agent, group in data.groupby("agent"):

        values = np.sort(
            group["ttm_hours"].astype(float).to_numpy()
        )

        q_values = np.quantile(
            values,
            quantiles
        )

        row = {
            "agent": agent,
            "n": len(values)
        }

        for name, value in zip(
            quantile_names,
            q_values
        ):
            row[name] = float(value)

        summary.append(row)

        # Limita a ECDF a 300 pontos por agente
        max_points = 300

        indices = np.unique(
            np.linspace(
                0,
                len(values) - 1,
                min(len(values), max_points)
            ).round().astype(int)
        )

        for index in indices:

            ecdf.append({
                "agent": agent,
                "ttm_hours": float(values[index]),
                "probability": float(
                    (index + 1) / len(values)
                )
            })

    return {
        "summary": summary,
        "ecdf": ecdf
    }


merge_times = prepare_merge_times(pr)

write_json(
    OUTPUT_DIR / "merge_times.json",
    merge_times
)

print("merge_times.json gerado!")

merge_times.json gerado!


In [63]:
def prepare_calendar(pr):

    data = pr.dropna(
        subset=["date"]
    ).copy()

    if data.empty:
        return []

    daily = (
        data.groupby(
            ["date", "agent"],
            observed=True
        )
        .size()
        .unstack(fill_value=0)
    )

    all_dates = pd.date_range(
        data["date"].min(),
        data["date"].max(),
        freq="D"
    )

    daily = daily.reindex(
        all_dates,
        fill_value=0
    )

    agents = sorted(
        data["agent"].dropna().unique()
    )

    result = []

    for day, row in daily.iterrows():

        agent_counts = {
            agent: int(row.get(agent, 0))
            for agent in agents
        }

        result.append({
            "date": day.strftime("%Y-%m-%d"),

            "count": int(
                sum(agent_counts.values())
            ),

            "dow": int(day.dayofweek),

            "week": (
                day - pd.to_timedelta(
                    day.dayofweek,
                    unit="D"
                )
            ).strftime("%Y-%m-%d"),

            "agents": agent_counts
        })

    return result


calendar_data = prepare_calendar(pr)

write_json(
    OUTPUT_DIR / "calendar.json",
    calendar_data
)

print("calendar.json gerado!")

calendar.json gerado!


In [64]:
def prepare_languages(pr):

    data = pr.dropna(
        subset=["language", "agent", "outcome"]
    ).copy()

    result = (
        data.groupby(
            ["language", "agent", "outcome"],
            observed=True
        )
        .size()
        .reset_index(name="count")
    )

    result["total"] = (
        result.groupby("language")["count"]
        .transform("sum")
    )

    result["percentage"] = (
        result["count"] /
        result["total"] * 100
    )

    return records(
        result.sort_values(
            ["language", "agent", "outcome"]
        )
    )


languages = prepare_languages(pr)

write_json(
    OUTPUT_DIR / "languages.json",
    languages
)

print("languages.json gerado!")

languages.json gerado!


In [65]:
def prepare_repositories(repos):

    result = repos.sort_values(
        "n_pr",
        ascending=False
    ).copy()

    return records(result)


repositories = prepare_repositories(repos)

write_json(
    OUTPUT_DIR / "repositories.json",
    repositories
)

print("repositories.json gerado!")

repositories.json gerado!


In [66]:
def prepare_network(pr, repos, top_n=100):

    # Seleciona os repositórios mais ativos
    top_repo_ids = (
        repos.sort_values(
            "n_pr",
            ascending=False
        )
        .head(top_n)["repo_id"]
        .dropna()
        .tolist()
    )

    repo_lookup = (
        repos.drop_duplicates("repo_id")
        .set_index("repo_id")["repo"]
        .to_dict()
    )

    # Conta PRs por agente e repositório
    edges_df = (
        pr.loc[
            pr["repo_id"].isin(top_repo_ids)
        ]
        .dropna(
            subset=["agent", "repo_id"]
        )
        .groupby(
            ["agent", "repo_id"],
            observed=True
        )
        .size()
        .reset_index(name="prs")
    )

    agents = sorted(
        edges_df["agent"].unique().tolist()
    )

    used_repo_ids = set(
        edges_df["repo_id"].tolist()
    )

    # Criação dos nós dos agentes
    nodes = [
        {
            "id": f"agent:{agent}",
            "label": agent,
            "type": "agent"
        }
        for agent in agents
    ]

    # Criação dos nós dos repositórios
    for repo_id in sorted(
        used_repo_ids,
        key=lambda x: str(x)
    ):

        nodes.append({
            "id": f"repo:{repo_id}",
            "label": repo_lookup.get(
                repo_id,
                str(repo_id)
            ),
            "type": "repository",
            "repo_id": json_safe(repo_id)
        })

    # Criação das arestas
    edges = []

    for row in edges_df.itertuples(index=False):

        edges.append({
            "source": f"agent:{row.agent}",
            "target": f"repo:{row.repo_id}",
            "prs": int(row.prs)
        })

    return {
        "nodes": nodes,
        "edges": edges,
        "config": {
            "top_repositories": int(top_n),
            "description": (
                "Rede bipartida entre agentes e repositórios. "
                "O peso da aresta representa a quantidade de PRs."
            )
        }
    }


network = prepare_network(
    pr,
    repos,
    NETWORK_TOP
)

write_json(
    OUTPUT_DIR / "agent_repository_network.json",
    network
)

print("agent_repository_network.json gerado!")
print(f"Nós: {len(network['nodes']):,}")
print(f"Arestas: {len(network['edges']):,}")

agent_repository_network.json gerado!
Nós: 106
Arestas: 178


In [67]:
agents = sorted(
    pr["agent"].dropna().unique().tolist()
)

languages_list = sorted(
    pr["language"].dropna().unique().tolist()
)

outcomes_list = sorted(
    pr["outcome"].dropna().unique().tolist()
)

review_types = sorted(
    pr["review_type"].dropna().unique().tolist()
)

date_min = pr["date"].min()
date_max = pr["date"].max()

output_meta = {

    "total_prs": int(len(pr)),

    "total_repositories": int(
        repos["repo_id"].nunique()
    ),

    "agents": agents,

    "languages": languages_list,

    "outcomes": outcomes_list,

    "review_types": review_types,

    "stars_bins": source_meta.get(
        "stars_bins",
        []
    ),

    "date_min": (
        date_min.strftime("%Y-%m-%d")
        if pd.notna(date_min)
        else None
    ),

    "date_max": (
        date_max.strftime("%Y-%m-%d")
        if pd.notna(date_max)
        else None
    ),

    "network_top_repositories": NETWORK_TOP,

    "source_metadata": source_meta,

    "methodological_notes": [

        "Os dados representam o recorte fornecido em pr_slim.csv e repos.csv.",

        "O tempo até merge considera somente PRs com outcome merged e ttm_hours positivo.",

        "A rede é bipartida: agente-repositório.",

        "O peso da aresta representa a quantidade de Pull Requests.",

        "A rede considera os repositórios mais ativos definidos por NETWORK_TOP.",

        "calendar.json inclui dias sem PRs no intervalo, com contagem zero.",

        "Os percentuais em outcomes.json são calculados dentro de cada agente.",

        "Os percentuais em languages.json são calculados dentro de cada linguagem."
    ]
}

write_json(
    OUTPUT_DIR / "meta.json",
    output_meta
)

print("meta.json gerado!")

meta.json gerado!


In [68]:
agents = sorted(
    pr["agent"].dropna().unique().tolist()
)

languages_list = sorted(
    pr["language"].dropna().unique().tolist()
)

outcomes_list = sorted(
    pr["outcome"].dropna().unique().tolist()
)

review_types = sorted(
    pr["review_type"].dropna().unique().tolist()
)

date_min = pr["date"].min()
date_max = pr["date"].max()

output_meta = {

    "total_prs": int(len(pr)),

    "total_repositories": int(
        repos["repo_id"].nunique()
    ),

    "agents": agents,

    "languages": languages_list,

    "outcomes": outcomes_list,

    "review_types": review_types,

    "stars_bins": source_meta.get(
        "stars_bins",
        []
    ),

    "date_min": (
        date_min.strftime("%Y-%m-%d")
        if pd.notna(date_min)
        else None
    ),

    "date_max": (
        date_max.strftime("%Y-%m-%d")
        if pd.notna(date_max)
        else None
    ),

    "network_top_repositories": NETWORK_TOP,

    "source_metadata": source_meta,

    "methodological_notes": [

        "Os dados representam o recorte fornecido em pr_slim.csv e repos.csv.",

        "O tempo até merge considera somente PRs com outcome merged e ttm_hours positivo.",

        "A rede é bipartida: agente-repositório.",

        "O peso da aresta representa a quantidade de Pull Requests.",

        "A rede considera os repositórios mais ativos definidos por NETWORK_TOP.",

        "calendar.json inclui dias sem PRs no intervalo, com contagem zero.",

        "Os percentuais em outcomes.json são calculados dentro de cada agente.",

        "Os percentuais em languages.json são calculados dentro de cada linguagem."
    ]
}

write_json(
    OUTPUT_DIR / "meta.json",
    output_meta
)

print("meta.json gerado!")

meta.json gerado!


In [69]:

print("ARQUIVOS JSON GERADOS:\n")

for arquivo in sorted(OUTPUT_DIR.glob("*.json")):
    print(f"{arquivo.name} - {arquivo.stat().st_size:,} bytes")

print("\nTotal:", len(list(OUTPUT_DIR.glob("*.json"))), "arquivos")


ARQUIVOS JSON GERADOS:

agent_repository_network.json - 32,807 bytes
calendar.json - 75,574 bytes
languages.json - 37,732 bytes
merge_times.json - 197,425 bytes
meta.json - 2,119 bytes
monthly_agents.json - 3,526 bytes
outcomes.json - 2,597 bytes
repositories.json - 1,915,448 bytes

Total: 8 arquivos
